# A bigger factorial: four factors and model reduction

**Source worksheet:** [yint.org/w5](https://yint.org/w5) - week 5 of the applied DoE short course.

Module 3 finished with a 2^3 cube.  This module steps up to four
factors and 16 runs.  The mathematics is unchanged - it is still
``y = b_0 + sum(b_i * x_i) + sum(b_ij * x_i * x_j) + ...`` - but two
new habits matter:

1. **Count the terms before you fit.**  With ``k`` factors you have
   ``2^k`` runs and ``2^k`` terms in the full polynomial: 1 intercept,
   ``k`` main effects, ``C(k,2)`` two-factor interactions, and so on.
   This sets the ``R^2`` budget: a full model on a saturated design
   *must* hit ``R^2 = 1`` exactly because there is one coefficient
   per observation.
2. **Reduce the model.**  Real responses depend on a handful of
   factors and interactions; the rest of the polynomial estimates
   noise.  Reading the *Pareto* of effect magnitudes and dropping
   the small ones recovers degrees of freedom and produces a model
   you can trust.

## Q1-Q2 - counting interactions

A 2^4 design has:

- ``4`` main effects,
- ``C(4, 2) = 6`` two-factor interactions,
- ``C(4, 3) = 4`` three-factor interactions,
- ``C(4, 4) = 1`` four-factor interaction.

Plus one intercept, that is 16 coefficients in the full model.  With
16 runs and 16 coefficients there are zero residual degrees of freedom;
``R^2`` is exactly 1 and the residual standard error is exactly 0.
This is not a sign of a great model - it is a sign that the model has
no room left to disagree with the data.

In [ ]:
from math import comb

k = 4
print(f"Main effects     : {k}")
print(f"2-factor interactions: {comb(k, 2)}")
print(f"3-factor interactions: {comb(k, 3)}")
print(f"4-factor interactions: {comb(k, 4)}")
print(f"Total terms (incl intercept): {sum(comb(k, j) for j in range(k + 1))}")
print(f"Runs available  : {2 ** k}")
print(f"Residual DoF    : {2 ** k - sum(comb(k, j) for j in range(k + 1))}")

## Q3 - the bioreactor system

Four factors are varied in 16 random-order runs:

- **A = feed rate** [5 or 8 g/min]
- **B = initial inoculant** [300 or 400 g]
- **C = feed substrate concentration** [40 or 60 g/L]
- **D = dissolved oxygen set-point** [4 or 5 mg/L]

In standard order (A varies fastest, then B, then C, then D), the
yields (in g/L) are:

```
y = [60, 59, 63, 61, 69, 61, 94, 93,
     56, 63, 70, 65, 44, 45, 78, 77]
```

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

from process_improve.experiments import c, gather, lm, predict
from process_improve.experiments.visualization import visualize_doe

pio.renderers.default = "notebook_connected"

# Standard order: A flips every row, B every 2, C every 4, D every 8.

A = c(-1, +1, -1, +1, -1, +1, -1, +1, -1, +1, -1, +1, -1, +1, -1, +1, name="A", lo=5,   hi=8)
B = c(-1, -1, +1, +1, -1, -1, +1, +1, -1, -1, +1, +1, -1, -1, +1, +1, name="B", lo=300, hi=400)
C = c(-1, -1, -1, -1, +1, +1, +1, +1, -1, -1, -1, -1, +1, +1, +1, +1, name="C", lo=40,  hi=60)
D = c(-1, -1, -1, -1, -1, -1, -1, -1, +1, +1, +1, +1, +1, +1, +1, +1, name="D", lo=4,   hi=5)
y = c(60, 59, 63, 61, 69, 61, 94, 93,
      56, 63, 70, 65, 44, 45, 78, 77, name="y")
bio = gather(A=A, B=B, C=C, D=D, y=y)
print(f"Design has {len(bio)} runs and {bio.shape[1] - 1} factors")

In [ ]:
# Fit the full 4-way model.  Every interaction term is included; the
# design is saturated so R^2 = 1 by construction.

full = lm("y ~ A * B * C * D", bio)
params = full.get_parameters(drop_intercept=False)

# Sort by absolute magnitude to spot the dominant terms.

ordered = params.reindex(params.abs().sort_values(ascending=False).index)
print(ordered.to_string())

In [ ]:
# Pareto-style bar chart of the effects (= 2 * coefficient).

effects = {
    term: 2 * coef
    for term, coef in full.get_parameters(drop_intercept=True).items()
}
pareto = visualize_doe(
    plot_type="pareto",
    analysis_results={"effects": effects},
    backend="plotly",
)
fig = go.Figure(pareto["plotly"])
fig.update_layout(width=720, height=460, title="Pareto plot of effects on bioreactor yield")
fig

## Q6 - reducing the model

Dropping A buys back four degrees of freedom (the main effect of A
plus the three-, two-, and one-way interactions that involve A).  The
*orthogonal* design guarantees the coefficients on the remaining
terms are unchanged - only the standard errors and the residual
degrees of freedom move.

In [ ]:
reduced = lm("y ~ B * C * D", bio)
print("Reduced model (A and all its interactions dropped):")
print(reduced.get_parameters(drop_intercept=False).to_string())
print()
print(f"Reduced R^2: {reduced._OLS.rsquared:.4f}")
print(f"Reduced residual std error: {(reduced._OLS.scale ** 0.5):.2f} g/L")

## Q7-Q8 - where to go next

To **maximize** yield:

- ``B`` is positive -> set B high (+1).
- ``C`` is positive **and** ``B:C`` is positive -> set C high (+1) too;
  the interaction amplifies B.
- ``D`` is negative **and** ``C:D`` is negative -> set D low (-1);
  high D actively hurts when C is high.

That points to the corner ``B = +1, C = +1, D = -1`` (with A wherever
is cheapest).  Several extrapolation candidates are explored below.

In [ ]:
candidates = [
    ("B=+1, C=+1, D=-1 (corner)",       dict(A=0, B=+1,   C=+1,   D=-1)),
    ("B=+1.5, C=+1.5, D=-1",             dict(A=0, B=+1.5, C=+1.5, D=-1)),
    ("B=+2,   C=+1,   D=-1.5",           dict(A=0, B=+2,   C=+1,   D=-1.5)),
    ("B=+2,   C=+2,   D=-1",             dict(A=0, B=+2,   C=+2,   D=-1)),
]
for label, point in candidates:
    pred = float(predict(reduced, **point).iloc[0])
    print(f"{label:40s} -> predicted yield {pred:5.1f} g/L")

## Wrap-up

Two transferable lessons that scale to any number of factors:

- A **saturated full factorial** spends every observation on a
  coefficient.  The model fits perfectly and tells you nothing about
  uncertainty.  Replicates (Module 3) or model reduction (this
  module) give you that uncertainty back.
- **Orthogonality** is what makes "just drop the small terms" safe.
  In a designed full factorial the columns of the model matrix are
  uncorrelated, so removing a factor does not bias the remaining
  coefficients.

**Next:** Module 5 attacks the same problem from a different direction.
What if you cannot afford 16 runs to begin with?  Fractional
factorials trade *resolution* for *budget*: half a factorial costs
half as much, and (usually) costs only the highest-order interaction
you did not believe in anyway.